# Home Credit: PSI → PSI-CA → Private Intersection-Sum (PIS)

Единый ноутбук на **полном объёме данных**, адаптированный из `Untitled2.ipynb` коллеги. Раздел с искусственным тестом на 100 записях удалён.

**Три вычисления:** (1) OpenMined PSI — ID пересечения получает банк, (2) OpenMined PSI-CA — банк получает только число, (3) Google Private Join and Compute — банк получает размер пересечения и **сумму `TARGET` по общим клиентам**, без списка ID. В последнем разделе используется настоящая отдельная реализация PIS на C++ (через Bazel), а не незащищённое пересечение Pandas.

Среда: **Google Colab (Linux)**; нужны файлы `/content/bank_psi.parquet` и `/content/telecom_train.parquet`. Раздел PIS требует Git, Bazelisk и сборки C++: первая сборка может быть тяжёлой, особенно на бесплатном Colab. Полные вычисления могут занять продолжительное время. **Никаких подвыборок**.

⚠️ Данные этого эксперимента происходят из одного набора Home Credit: разделение сторон моделируется. Протоколы в ноутбуке выполняются в одной Colab-среде; для реального разделения доверия банк и телеком должны исполнять роли в разных окружениях.

## 0. Подготовка среды и полный объём исходных данных

In [1]:
%pip -q install openmined-psi pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import time
import pandas as pd
import pyarrow.parquet as pq
import private_set_intersection.python as psi

BANK_PATH = Path('../data/bank_psi.parquet')
TELECOM_PATH = Path('../data/telecom_train.parquet')

for p in (BANK_PATH, TELECOM_PATH):
    if not p.exists():
        raise FileNotFoundError(f'Отсутствует {p}. Загрузите файл в /content/')
    pf = pq.ParquetFile(p)
    print(f'{p.name}: {pf.metadata.num_rows:,} строк; {pf.metadata.num_columns} столбцов')

bank_df = pd.read_parquet(BANK_PATH, columns=['SK_ID_CURR', 'TARGET']).dropna(subset=['SK_ID_CURR', 'TARGET']).copy()
telecom_df = pd.read_parquet(TELECOM_PATH, columns=['SK_ID_CURR']).dropna(subset=['SK_ID_CURR']).copy()

# Уникальный клиент -> одна цель. Если есть конфликтующие TARGET, прерываемся.
if bank_df.groupby('SK_ID_CURR')['TARGET'].nunique().gt(1).any():
    raise ValueError('Обнаружены конфликтующие TARGET для одного SK_ID_CURR')
bank_df = bank_df.drop_duplicates('SK_ID_CURR')
telecom_df = telecom_df.drop_duplicates('SK_ID_CURR')
if not bank_df['TARGET'].isin([0, 1]).all():
    raise ValueError('TARGET должен быть бинарным (0/1)')

bank_df['ID'] = bank_df['SK_ID_CURR'].astype('int64').astype(str)
telecom_df['ID'] = telecom_df['SK_ID_CURR'].astype('int64').astype(str)
bank_new = sorted(bank_df['ID'].tolist())
telecom_new = sorted(telecom_df['ID'].tolist())

# Эталон рассчитывается локально только для проверки учебного эксперимента.
# При реальных двух организациях этот открытый JOIN недопустим.
bank_target_map = dict(zip(bank_df['ID'], bank_df['TARGET'].astype(int)))
expected_new = set(bank_new) & set(telecom_new)
expected_sum = sum(bank_target_map[i] for i in expected_new)

print('Банк:', len(bank_new))
print('Телеком:', len(telecom_new))
print('Эталонное пересечение:', len(expected_new))
print('Эталонная сумма TARGET:', expected_sum)
print('Только банк:', len(set(bank_new) - set(telecom_new)))
print('Только телеком:', len(set(telecom_new) - set(bank_new)))

bank_psi.parquet: 307,491 строк; 382 столбцов
telecom_train.parquet: 307,511 строк; 6 столбцов
Банк: 307491
Телеком: 307511
Эталонное пересечение: 307491
Эталонная сумма TARGET: 24824
Только банк: 0
Только телеком: 20


## 1. PSI — банк получает список общих ID

Сохранена логика коллеги на всей выборке, включая проверку с обычным пересечением. Здесь намеренно раскрывается список ID стороне банка.

In [3]:
print('Запуск полного PSI...')
t0 = time.perf_counter()
bank_client = psi.client.CreateWithNewKey(True)
telecom_server = psi.server.CreateWithNewKey(True)
server_setup = telecom_server.CreateSetupMessage(
    0.001, len(bank_new), telecom_new, psi.DataStructure.RAW
)
request = bank_client.CreateRequest(bank_new)
response = telecom_server.ProcessRequest(request)
indices = bank_client.GetIntersection(server_setup, response)
bank_result_new = [bank_new[i] for i in indices]
psi_seconds = time.perf_counter() - t0

assert set(bank_result_new) == expected_new
print('PSI: совпадений', len(bank_result_new))
print(f'Время: {psi_seconds:.2f} с')
print('Проверка: OK')

Запуск полного PSI...
PSI: совпадений 307491
Время: 89.44 с
Проверка: OK


## 2. PSI-CA — банк получает только количество общих ID

Тот же протокол OpenMined, но флаг `False` и метод `GetIntersectionSize()`; банк не получает индексы совпадений из этого запуска. Используется тот же полный объём и структура `RAW`.

In [4]:
print('Запуск полного PSI-CA...')
t0 = time.perf_counter()
bank_client_ca = psi.client.CreateWithNewKey(False)
telecom_server_ca = psi.server.CreateWithNewKey(False)
setup_ca = telecom_server_ca.CreateSetupMessage(
    0.001, len(bank_new), telecom_new, psi.DataStructure.RAW
)
request_ca = bank_client_ca.CreateRequest(bank_new)
response_ca = telecom_server_ca.ProcessRequest(request_ca)
intersection_size_ca = bank_client_ca.GetIntersectionSize(setup_ca, response_ca)
psi_ca_seconds = time.perf_counter() - t0

assert intersection_size_ca == len(expected_new)
print('PSI-CA: общих клиентов', intersection_size_ca)
print(f'Время: {psi_ca_seconds:.2f} с')
print('Проверка: OK')

Запуск полного PSI-CA...
PSI-CA: общих клиентов 307491
Время: 87.42 с
Проверка: OK
